# 🛡️ CyberDefense-AI — NSL-KDD Model Training
**Dataset:** NSL-KDD (148,517 rows · 41 features)  
**Kernel:** CyberDefense AI  

Supplementary model for DoS, Port Scan, Brute Force, Infiltration detection.
Trains in ~5 minutes. Good cross-validation of the main CICIDS models.

### Output files saved to `backend/models/`
| File | Description |
|------|-------------|
| `network_nslkdd_model.pkl` | RF classifier (41 features) |
| `network_nslkdd_dl.pth` | DNN weights |
| `network_nslkdd_scaler.pkl` | StandardScaler |
| `network_nslkdd_encoder.pkl` | LabelEncoder |
| `network_nslkdd_shap.json` | SHAP values |
| `network_nslkdd_info.json` | Metadata |

### Attack classes
`BENIGN · PORT_SCAN · BRUTE_FORCE · DOS · INFILTRATION`

In [ ]:
import os, json, pickle, warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.utils import resample
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

import torch, torch.nn as nn, torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import shap

plt.style.use('dark_background')
NEON_RED='#ff003c'; NEON_BLUE='#00d4ff'; NEON_GREEN='#00ff88'; NEON_ORANGE='#ff8c00'
CLASS_COLORS = {
    'BENIGN':'#00ff88','PORT_SCAN':'#00d4ff','BRUTE_FORCE':'#ff8c00',
    'DOS':'#ff003c','INFILTRATION':'#a78bfa',
}

BASE       = r'C:\Users\Gagandeep Singh\Desktop\Active projects\cyberdefense-platform'
DATA_NSL   = os.path.join(BASE, 'notebooks', 'data', 'NSL-KDD')
MODELS_OUT = os.path.join(BASE, 'backend', 'models')
PLOTS_OUT  = os.path.join(BASE, 'notebooks')
os.makedirs(MODELS_OUT, exist_ok=True)

print(f'torch  {torch.__version__}')
print(f'GPU    {torch.cuda.is_available()}')
print('✅ Imports OK')

---
## Phase 1 — Load NSL-KDD

In [ ]:
NSL_COLS = [
    'duration','protocol_type','service','flag','src_bytes','dst_bytes',
    'land','wrong_fragment','urgent','hot','num_failed_logins','logged_in',
    'num_compromised','root_shell','su_attempted','num_root','num_file_creations',
    'num_shells','num_access_files','num_outbound_cmds','is_host_login',
    'is_guest_login','count','srv_count','serror_rate','srv_serror_rate',
    'rerror_rate','srv_rerror_rate','same_srv_rate','diff_srv_rate',
    'srv_diff_host_rate','dst_host_count','dst_host_srv_count',
    'dst_host_same_srv_rate','dst_host_diff_srv_rate',
    'dst_host_same_src_port_rate','dst_host_srv_diff_host_rate',
    'dst_host_serror_rate','dst_host_srv_serror_rate',
    'dst_host_rerror_rate','dst_host_srv_rerror_rate',
    'label','difficulty'
]

df_train = pd.read_csv(os.path.join(DATA_NSL,'KDDTrain+.txt'), header=None, names=NSL_COLS)
df_test  = pd.read_csv(os.path.join(DATA_NSL,'KDDTest+.txt'),  header=None, names=NSL_COLS)
df       = pd.concat([df_train, df_test], ignore_index=True)

print(f'Train  : {len(df_train):,}')
print(f'Test   : {len(df_test):,}')
print(f'Total  : {len(df):,}')
print(f'Columns: {len(df.columns)}')
print('\nRaw label distribution (top 20):')
print(df['label'].value_counts().head(20))

In [ ]:
# ── Label mapping ─────────────────────────────────────────
LABEL_MAP = {
    'normal':           'BENIGN',
    # DoS
    'back':'DOS','land':'DOS','neptune':'DOS','pod':'DOS','smurf':'DOS',
    'teardrop':'DOS','apache2':'DOS','udpstorm':'DOS','processtable':'DOS','worm':'DOS',
    # Probe / Port Scan
    'ipsweep':'PORT_SCAN','mscan':'PORT_SCAN','nmap':'PORT_SCAN',
    'portsweep':'PORT_SCAN','saint':'PORT_SCAN','satan':'PORT_SCAN',
    # R2L / Brute Force
    'ftp_write':'BRUTE_FORCE','guess_passwd':'BRUTE_FORCE','httptunnel':'BRUTE_FORCE',
    'imap':'BRUTE_FORCE','multihop':'BRUTE_FORCE','named':'BRUTE_FORCE',
    'phf':'BRUTE_FORCE','sendmail':'BRUTE_FORCE','snmpgetattack':'BRUTE_FORCE',
    'snmpguess':'BRUTE_FORCE','spy':'BRUTE_FORCE','warezclient':'BRUTE_FORCE',
    'warezmaster':'BRUTE_FORCE','xlock':'BRUTE_FORCE','xsnoop':'BRUTE_FORCE',
    # U2R / Privilege escalation
    'buffer_overflow':'INFILTRATION','loadmodule':'INFILTRATION','perl':'INFILTRATION',
    'ps':'INFILTRATION','rootkit':'INFILTRATION','sqlattack':'INFILTRATION','xterm':'INFILTRATION',
}

df['attack_class'] = df['label'].map(lambda x: LABEL_MAP.get(x.strip(), 'BRUTE_FORCE'))

# Encode categorical features
for col in ('protocol_type','service','flag'):
    df[col] = LabelEncoder().fit_transform(df[col].astype(str))

FEAT_COLS = NSL_COLS[:-2]   # 41 features
df[FEAT_COLS] = df[FEAT_COLS].apply(pd.to_numeric, errors='coerce').fillna(0)

print('Mapped class distribution:')
print(df['attack_class'].value_counts())
print(f'\nFeatures: {len(FEAT_COLS)}')

In [ ]:
# Class distribution plot
counts = df['attack_class'].value_counts()
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
fig.patch.set_facecolor('#000010')

colors = [CLASS_COLORS.get(c,'#888') for c in counts.index]

ax1.set_facecolor('#050510')
bars = ax1.bar(counts.index, counts.values, color=colors, edgecolor='black', linewidth=0.5)
ax1.set_title('NSL-KDD Class Distribution', color='#00d4ff', fontsize=13)
ax1.tick_params(colors='white', labelsize=9)
for bar, v in zip(bars, counts.values):
    ax1.text(bar.get_x()+bar.get_width()/2., bar.get_height()*1.01,
             f'{v:,}', ha='center', va='bottom', color='white', fontsize=9)

ax2.set_facecolor('#050510')
ax2.pie(counts.values, labels=counts.index, colors=colors,
        autopct='%1.1f%%', textprops={'color':'white','fontsize':9}, startangle=90)
ax2.set_title('Attack Proportions', color='#00d4ff', fontsize=13)

plt.tight_layout()
plt.savefig(os.path.join(PLOTS_OUT,'nsl_kdd_distribution.png'),
            dpi=120, bbox_inches='tight', facecolor='#000010')
plt.show()

---
## Phase 2 — Balance + Encode + Split

In [ ]:
TARGET = 40_000
MIN    = 2_000

balanced = []
for cls in df['attack_class'].unique():
    c = df[df['attack_class'] == cls]
    n = len(c)
    if n > TARGET:
        c = resample(c, n_samples=TARGET, replace=False, random_state=42)
    elif n < MIN:
        c = resample(c, n_samples=MIN,    replace=True,  random_state=42)
    balanced.append(c)

df_bal = pd.concat(balanced, ignore_index=True).sample(frac=1, random_state=42)
print('Balanced distribution:')
print(df_bal['attack_class'].value_counts())
print(f'Total: {len(df_bal):,}')

le    = LabelEncoder()
y     = le.fit_transform(df_bal['attack_class'])
X     = df_bal[FEAT_COLS].values.astype(np.float32)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=42
)

scaler    = StandardScaler()
X_tr_sc   = scaler.fit_transform(X_train)
X_te_sc   = scaler.transform(X_test)

print(f'\nClasses : {le.classes_.tolist()}')
print(f'Train   : {X_train.shape}')
print(f'Test    : {X_test.shape}')

---
## Phase 3 — Random Forest

In [ ]:
print('Training Random Forest...')
rf_nsl = RandomForestClassifier(
    n_estimators  = 200,
    max_features  = 'sqrt',
    class_weight  = 'balanced_subsample',
    n_jobs        = -1,
    random_state  = 42,
    verbose       = 1
)
rf_nsl.fit(X_train, y_train)

y_pred = rf_nsl.predict(X_test)
rf_acc = accuracy_score(y_test, y_pred)
rf_f1  = f1_score(y_test, y_pred, average='weighted')

print(f'\n✅ RF Accuracy : {rf_acc*100:.2f}%')
print(f'   RF F1       : {rf_f1*100:.2f}%')
print()
print(classification_report(y_test, y_pred, target_names=le.classes_))

In [ ]:
cm = confusion_matrix(y_test, y_pred)
fig, ax = plt.subplots(figsize=(8,6))
fig.patch.set_facecolor('#000010'); ax.set_facecolor('#050510')
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=le.classes_, yticklabels=le.classes_,
            linewidths=0.3, linecolor='#111', ax=ax)
ax.set_title('NSL-KDD Confusion Matrix', color='#00d4ff', fontsize=13)
ax.set_xlabel('Predicted', color='#666'); ax.set_ylabel('Actual', color='#666')
ax.tick_params(colors='white', labelsize=9)
plt.setp(ax.xaxis.get_majorticklabels(), rotation=30, ha='right')
plt.tight_layout()
plt.savefig(os.path.join(PLOTS_OUT,'nsl_kdd_confusion.png'),
            dpi=120, bbox_inches='tight', facecolor='#000010')
plt.show()

---
## Phase 4 — PyTorch DNN

In [ ]:
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
NC = len(le.classes_); ID = len(FEAT_COLS)

class NSLNet(nn.Module):
    def __init__(self, in_d, n_cls):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(in_d, 256), nn.BatchNorm1d(256), nn.ReLU(), nn.Dropout(0.30),
            nn.Linear(256, 128),  nn.BatchNorm1d(128), nn.ReLU(), nn.Dropout(0.25),
            nn.Linear(128, 64),   nn.BatchNorm1d(64),  nn.ReLU(), nn.Dropout(0.20),
            nn.Linear(64,  32),   nn.ReLU(),
            nn.Linear(32,  n_cls)
        )
    def forward(self, x): return self.network(x)

nsl_dl = NSLNet(ID, NC).to(DEVICE)
cw     = compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
crit   = nn.CrossEntropyLoss(weight=torch.FloatTensor(cw).to(DEVICE))
opt    = optim.AdamW(nsl_dl.parameters(), lr=5e-4, weight_decay=5e-4)

ds  = TensorDataset(torch.FloatTensor(X_tr_sc).to(DEVICE),
                    torch.LongTensor(y_train).to(DEVICE))
dl  = DataLoader(ds, batch_size=512, shuffle=True)
sch = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=30, eta_min=1e-5)

best_acc, best_st = 0.0, None
losses, accs = [], []

for ep in range(1, 31):
    nsl_dl.train()
    ep_loss = 0
    for xb, yb in dl:
        opt.zero_grad()
        loss = crit(nsl_dl(xb), yb)
        loss.backward()
        nn.utils.clip_grad_norm_(nsl_dl.parameters(), 1.0)
        opt.step()
        ep_loss += loss.item()
    sch.step()
    nsl_dl.eval()
    with torch.no_grad():
        p   = nsl_dl(torch.FloatTensor(X_te_sc).to(DEVICE)).argmax(1).cpu().numpy()
        acc = accuracy_score(y_test, p) * 100
    losses.append(ep_loss/len(dl)); accs.append(acc)
    if acc > best_acc:
        best_acc = acc
        best_st  = {k: v.clone() for k, v in nsl_dl.state_dict().items()}
    if ep % 5 == 0 or ep == 1:
        print(f'Epoch {ep:2d}/30 | Loss: {ep_loss/len(dl):.4f} | Acc: {acc:.2f}% | Best: {best_acc:.2f}%')

if best_st:
    nsl_dl.load_state_dict(best_st)

nsl_dl.eval()
with torch.no_grad():
    y_dl = nsl_dl(torch.FloatTensor(X_te_sc).to(DEVICE)).argmax(1).cpu().numpy()
dl_acc = accuracy_score(y_test, y_dl)
dl_f1  = f1_score(y_test, y_dl, average='weighted')

print(f'\n✅ DNN Accuracy : {dl_acc*100:.2f}%')
print(f'   DNN F1       : {dl_f1*100:.2f}%')
print()
print(classification_report(y_test, y_dl, target_names=le.classes_))

---
## Phase 5 — SHAP

In [ ]:
print('Computing SHAP values...')
exp    = shap.TreeExplainer(rf_nsl)
sv     = exp.shap_values(X_test[:1000])
sv_abs = np.mean([np.abs(s) for s in sv], axis=0) if isinstance(sv, list) else np.abs(sv)
shap_s = pd.Series(sv_abs.mean(axis=0), index=FEAT_COLS).sort_values(ascending=False)

top15 = shap_s.head(15)
fig, ax = plt.subplots(figsize=(10, 6))
fig.patch.set_facecolor('#000010'); ax.set_facecolor('#050510')
ax.barh(top15.index[::-1], top15.values[::-1], color=NEON_BLUE,
        edgecolor='black', linewidth=0.3)
ax.set_title('NSL-KDD SHAP Feature Importance', color='#00d4ff', fontsize=13)
ax.tick_params(colors='white', labelsize=9)
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.savefig(os.path.join(PLOTS_OUT,'nsl_kdd_shap.png'),
            dpi=120, bbox_inches='tight', facecolor='#000010')
plt.show()
print('Top 10 features:')
print(shap_s.head(10).to_string())

---
## Phase 6 — Save Models

In [ ]:
with open(os.path.join(MODELS_OUT,'network_nslkdd_model.pkl'),'wb') as f:   pickle.dump(rf_nsl, f)
with open(os.path.join(MODELS_OUT,'network_nslkdd_scaler.pkl'),'wb') as f:  pickle.dump(scaler, f)
with open(os.path.join(MODELS_OUT,'network_nslkdd_encoder.pkl'),'wb') as f: pickle.dump(le, f)
torch.save(nsl_dl.state_dict(), os.path.join(MODELS_OUT,'network_nslkdd_dl.pth'))
with open(os.path.join(MODELS_OUT,'network_nslkdd_shap.json'),'w') as f:    json.dump(shap_s.to_dict(), f, indent=2)

nsl_info = {
    'dataset':      'NSL-KDD',
    'total_samples': int(len(df_bal)),
    'features':     FEAT_COLS,
    'num_features': len(FEAT_COLS),
    'classes':      le.classes_.tolist(),
    'num_classes':  NC,
    'rf_accuracy':  round(rf_acc*100, 2),
    'rf_f1':        round(rf_f1*100,  2),
    'dl_accuracy':  round(dl_acc*100, 2),
    'dl_f1':        round(dl_f1*100,  2),
    'dnn_arch':     '256-128-64-32-NC',
    'top_features': shap_s.head(10).index.tolist(),
    'models': {
        'rf':      'network_nslkdd_model.pkl',
        'scaler':  'network_nslkdd_scaler.pkl',
        'encoder': 'network_nslkdd_encoder.pkl',
        'dl':      'network_nslkdd_dl.pth',
        'shap':    'network_nslkdd_shap.json',
    }
}
with open(os.path.join(MODELS_OUT,'network_nslkdd_info.json'),'w') as f:
    json.dump(nsl_info, f, indent=2)

print('=' * 50)
print('  NSL-KDD TRAINING COMPLETE')
print('=' * 50)
print(f'  RF  Accuracy : {rf_acc*100:.2f}%   F1: {rf_f1*100:.2f}%')
print(f'  DNN Accuracy : {dl_acc*100:.2f}%   F1: {dl_f1*100:.2f}%')
print(f'  Output       : {MODELS_OUT}')
print()
print('Files saved:')
for fn in sorted(os.listdir(MODELS_OUT)):
    if 'nslkdd' in fn:
        kb = os.path.getsize(os.path.join(MODELS_OUT,fn))/1024
        print(f'  {fn:<45} {kb:>8.0f} KB')

---
## Phase 7 — Verify

In [ ]:
with open(os.path.join(MODELS_OUT,'network_nslkdd_model.pkl'),'rb') as f:   rf_chk  = pickle.load(f)
with open(os.path.join(MODELS_OUT,'network_nslkdd_scaler.pkl'),'rb') as f:  sc_chk  = pickle.load(f)
with open(os.path.join(MODELS_OUT,'network_nslkdd_encoder.pkl'),'rb') as f: le_chk  = pickle.load(f)

dl_chk = NSLNet(len(FEAT_COLS), len(le_chk.classes_))
dl_chk.load_state_dict(torch.load(os.path.join(MODELS_OUT,'network_nslkdd_dl.pth'), map_location='cpu'))
dl_chk.eval()

sample_x  = X_test[:5]
sample_sc = sc_chk.transform(sample_x)

rf_pred = le_chk.classes_[rf_chk.predict(sample_x)]
with torch.no_grad():
    dl_pred = le_chk.classes_[dl_chk(torch.FloatTensor(sample_sc)).argmax(1).numpy()]
true_lbl = le_chk.classes_[y_test[:5]]

print(f'{"TRUE":<15} {"RF":<15} {"DNN"}')
print('-' * 45)
for t, r, d in zip(true_lbl, rf_pred, dl_pred):
    ok = '✅' if r == t else '❌'
    print(f'{ok} {t:<13} {r:<15} {d}')

print('\n✅ NSL-KDD models verified!')
print('\nAll outputs ready. Restart backend to activate ML detection.')